# Lesson 27: Debugging ML Models with Fairlearn, Cohort Error Analysis & SHAP

Replace proprietary dashboards with open-source **`fairlearn`**, decision-tree **Cohort Error Analysis**, and **`shap`** Global/Local explanations on Google Colab.

In [ ]:
!pip install -q fairlearn shap scikit-learn pandas

In [ ]:
import numpy as np
import pandas as pd
import shap
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.metrics import accuracy_score
from fairlearn.metrics import MetricFrame, selection_rate

rng = np.random.default_rng(27)
n = 300
X = pd.DataFrame({
    'age': rng.integers(25, 85, size=n),
    'time_in_hospital': rng.integers(1, 12, size=n),
    'prior_visits': rng.integers(0, 6, size=n),
    'group': rng.choice(['Cohort_A', 'Cohort_B'], size=n),
})
y = ((X['prior_visits'] >= 2) & (X['age'] > 55)).astype(int)
X_num = X[['age', 'time_in_hospital', 'prior_visits']]

clf = RandomForestClassifier(n_estimators=50, random_state=0).fit(X_num, y)
preds = clf.predict(X_num)

# 1. Fairlearn MetricFrame Disparity Check
mf = MetricFrame(metrics={'accuracy': accuracy_score, 'selection_rate': selection_rate}, y_true=y, y_pred=preds, sensitive_features=X['group'])
print(mf.by_group)

# 2. Cohort Error Analysis Tree
is_error = (y != preds).astype(int)
err_tree = DecisionTreeClassifier(max_depth=2, random_state=0).fit(X_num, is_error)
print('\nError Cohort Tree:\n', export_text(err_tree, feature_names=list(X_num.columns)))

# 3. SHAP Global & Local Explanations
explainer = shap.TreeExplainer(clf)
shap_values = explainer.shap_values(X_num.iloc[:10])
print('SHAP values computed for first 10 patients cleanly.')

## Assignment: Explore Responsible ML Debugging

Inspect `mf.difference()` across `time_in_hospital >= 7` vs `time_in_hospital < 3`, and print the local SHAP feature contributions for the oldest patient in `X_num`.